# Chapter 6 — Propositional Logic

**The Encyclopedia of Artificial Intelligence** — Perla Genovesi

This notebook contains the Python example from Chapter 6. Press the play button (▶) next to the code cell to run it: the result appears just below.

No installation is needed. The code uses only Python's standard library.

In [ ]:
from itertools import product

NAMES = {1: "A", 2: "B", 3: "C"}   # A, B, C: Anna, Bruno, Carla are in the office
KB = [[1, 2], [-1, -3], [-2, 3], [1, 3]]  # clauses: n atom, -n negation

def is_true(clauses, model):
    """A CNF is true if every clause has at least one true literal."""
    return all(any(model[abs(l)] == (l > 0) for l in c) for c in clauses)

def models(clauses, variables):
    """Truth table: all assignments that make the formula true."""
    for values in product([False, True], repeat=len(variables)):
        model = dict(zip(variables, values))
        if is_true(clauses, model):
            yield model

def simplify(clauses, assignment):
    """Remove true clauses and false literals; None if a clause is false."""
    new = []
    for c in clauses:
        if any(assignment.get(abs(l)) == (l > 0) for l in c):
            continue
        rest = [l for l in c if abs(l) not in assignment]
        if not rest:
            return None
        new.append(rest)
    return new

def dpll(clauses, assignment=None):
    """Return a model, or None if the formula is unsatisfiable."""
    assignment = dict(assignment or {})
    while True:                                    # unit propagation
        clauses = simplify(clauses, assignment)
        if clauses is None:
            return None
        if not clauses:
            return assignment
        units = [c[0] for c in clauses if len(c) == 1]
        if not units:
            break
        assignment[abs(units[0])] = units[0] > 0
    literal = clauses[0][0]  # choice and backtracking
    for value in (literal > 0, literal < 0):
        result = dpll(clauses, {**assignment, abs(literal): value})
        if result is not None:
            return result
    return None

def entails(kb, negated_alpha):
    """KB ⊨ α if and only if KB ∧ ¬α is unsatisfiable (¬α already as clauses)."""
    return dpll(kb + negated_alpha) is None

print("Models (truth table):")
for m in models(KB, [1, 2, 3]):
    print("  ", {NAMES[v]: m[v] for v in m})
model = dpll(KB)
print("Model found by DPLL:",
      {NAMES[v]: model[v] for v in sorted(model)})
print("KB ⊨ ¬(A ∧ B)?", entails(KB, [[1], [2]]))   # ¬α = A ∧ B
print("KB ⊨ A?", entails(KB, [[-1]]))              # ¬α = ¬A

## Expected output

If everything works, you should see:

```
Models (truth table):
   {'A': False, 'B': True, 'C': True}
   {'A': True, 'B': False, 'C': False}
Model found by DPLL: {'A': True, 'B': False, 'C': False}
KB ⊨ ¬(A ∧ B)? True
KB ⊨ A? False
```

Now try changing a number in the code and run it again to see what happens.